# TrappyTV Examplar notebook

In [ ]:
# Import Packages
import os
import sys
import numpy as np
import pandas as pd

In [ ]:
## Import Bokeh module and TrappyTV class
from bokeh.io import output_notebook
output_notebook()

from trappytv import TrappyTV

## Import Data

In [ ]:
#datapaths = ["data/merged_tracks.hd5"]
#datapaths = ["data/2026_01_02/m2_merged_tracks.hd5"]
datapaths = ["data/2025_12_30/M4_2025_12_30.hd5"]

In [ ]:
with pd.HDFStore("data/2025_12_30/M4_2025_12_30.hd5") as store:
    print(store.keys())

In [ ]:
class CellView: ## Datastructure to import 
    def __init__(self, datapath):
        if datapath.endswith(".hd5"):
            print("HDF datastore mode!")
            self.scopeid = "Trappy-Scope"
            self.paths = {"tracks": datapath}
        else:
            self.scopeid = os.path.basename(datapath)[:2]
            self.postprocess_path = os.path.join(datapath, "postprocess")
            self.paths = {"tracks": os.path.join(self.postprocess_path, "merged_tracks.hd5"),
                        "xyr_df": os.path.join(self.postprocess_path, "xyr.hd5")
                        }
            self.first_frames = np.load(os.path.join(self.postprocess_path, "first_frames.npy"))
        
        self.dfs = {key: pd.read_hdf(value, key="df") for key, value in self.paths.items()}
        try:
            self.metadata = pd.read_hdf(self.paths["tracks"], key="metadata")
        except:
            print("[WARNING] Metadata not found!")

        try:
            self.dfs["xyr_df"] = pd.read_hdf(self.paths["tracks"], key="xyr_df")
        except Exception as e:
            print(e)
            print("[WARNING] XYR data msiing")
        
        if "speed" not in self.dfs["tracks"].columns:
            self.dfs["tracks"]["speed"] = np.hypot(np.gradient(self.dfs["tracks"]["x_unrefined"]), np.gradient(self.dfs["tracks"]["y_unrefined"]))
        self.stuff = {}
    def __call__(self):
        return self.dfs["tracks"]

In [ ]:
cell = CellView(datapaths[0])
cell()

In [ ]:
cell.dfs["xyr_df"]

## View Data

In [ ]:
#%%time
TrappyTV(cell, width=1000, height=1000).view_all(sample=10)  ## Show the whole dataset but sample 10 points
#TrappyTV(cell, width=1000, height=1000).show(split_no=0)    ## Show only a particular split -> render all points